# Semantic Search for Web Archive Data

## Overview

This notebook demonstrates how to use LlamaIndex to build a data store for pre-processed HTML payloads extracted from web archive data. Then using that data store, perform semantic search over the web archive data.

### Building a Vector Store

Vector stores are a type of database designed to store and search embeddings, making them a straightforward and efficient way to implement semantic search.

1. Default vector store
2. FAISS vector store


### Semantic Search Pipeline

Perform semantic search using the prepared vector stores.

1. Load persisted vector stores (default or FAISS)
2. Configure query engines with different parameters
3. Run semantic search queries
4. Compare results and performance

## Environment Setup

### Installing Required Python Packages

In [ ]:
# Colab bootstrap: install the packages required for building the vector store and semantic search pipeline in this notebook. You can skip this cell if you are running the notebook in your local environment and have already installed the required packages.
%pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ wa-nlnz-toolkit
%pip install --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ wa-nlnz-semanticsearch

In [ ]:
# Configure environment based on execution context
try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Set output directory based on execution environment
res_folder = "/content" if IN_COLAB else "./"

## 1. Building a Vector Store

### Vector Stores

A vector store is a database designed specifically to store and efficiently search **embeddings (vectors)**. 

Traditional databases aren’t ideal for embeddings because they rely on exact matches, while embeddings require similarity-based search. A vector store addresses this by:
- Storing embeddings with their metadata  
- Enabling fast nearest-neighbor (similarity) search  
- Scaling efficiently to large datasets 


### Embeddings

Embeddings are numerical representations of data in the form of vectors. Instead of treating words or documents as plain strings, embeddings convert them into lists of numbers that capture their meaning and context.

The key idea is that similar pieces of content end up with similar vector representations. This allows machines to measure semantic similarity and understand relationships between concepts.

Typically, embeddings are generated using a pre-trained model that maps text into a high-dimensional vector space where meaning is preserved geometrically.

In [ ]:
import os
import numpy as np
import pandas as pd
from llama_index.core import Document, VectorStoreIndex, Settings, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


embedding_model = "all-MiniLM-L6-v2"  # Pre-trained embedding model
preprocessed_corpus_path = os.path.join(
    res_folder, "sample_data/preprocessed_corpus.parquet"
)

# ---------------------------
# 1. Load Model
# ---------------------------
Settings.llm = None
Settings.embed_model = HuggingFaceEmbedding(model_name=embedding_model)

# ----------------------------
# 2. Create Documents based on the loaded data
# ----------------------------
documents = [
    Document(
        text=row.text,
        metadata={
            "snapshot_id": str(row.snapshot_id),
            "page_idx": int(row.page_idx),
            "url": row.url,
        },
        excluded_llm_metadata_keys=["url"],
    )
    for row in pd.read_parquet(
        preprocessed_corpus_path,
        columns=["text", "snapshot_id", "page_idx", "url"],
    ).itertuples(index=False)
]

### Default vector store

Default in-memory vector store built into LlamaIndex, including

- Default Document Store (docstore)
- Default Vector Store (default__vector_store)
- Default Index (index_store)

It is essentially a Python-based vector database that:

- stores embeddings in lists / dicts
- performs similarity search using NumPy
- keeps everything in memory

In [ ]:
# ----------------------------
# Construct Index
# ----------------------------
index = VectorStoreIndex.from_documents(documents)
print("Indexed documents:", len(documents))

# ----------------------------
# Save index to disk
# ----------------------------
persist_dir = os.path.join(res_folder, "sample_data/vector_store/default")
os.makedirs(persist_dir, exist_ok=True)
index.storage_context.persist(persist_dir=persist_dir)

print(f"Vector store saved to {persist_dir}")

### FAISS vector store

A FAISS store in LlamaIndex is a wrapper around the FAISS (Facebook AI Similarity Search) library that acts as a vector database for storing and retrieving embeddings.

It provides:

- Efficient similarity search using FAISS algorithms (e.g., L2, cosine, IVF, HNSW)
- Fast nearest-neighbor lookup even for millions of vectors
- A connection layer between LlamaIndex’s documents/nodes and FAISS’s raw vector index

What it stores:

- The embedding vectors
- IDs that link vectors to LlamaIndex nodes/documents

What it does not store:

- Raw text
- Metadata
- Chunked nodes

Those are handled by LlamaIndex’s docstore, not FAISS.

In [ ]:
# Add additional import
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore


# Get embedding dimension from the model
embed_model = Settings.embed_model
# Create a sample embedding to get the dimension
sample_embedding = embed_model.get_text_embedding("sample")
embedding_dim = len(sample_embedding)

In [ ]:
# ----------------------------
# Create FAISS Index
# ----------------------------
# Create a FAISS index using L2 (Euclidean) distance
faiss_index = faiss.IndexFlatL2(embedding_dim)

# Wrap FAISS index with LlamaIndex's FaissVectorStore
vector_store = FaissVectorStore(faiss_index=faiss_index)

# Create storage context with FAISS vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# ----------------------------
# Construct Index with FAISS
# ----------------------------
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print("Indexed documents:", len(documents))

# ----------------------------
# Save index to disk
# ----------------------------
persist_dir = os.path.join(res_folder, "sample_data/vector_store/faiss")
os.makedirs(persist_dir, exist_ok=True)
index.storage_context.persist(persist_dir=persist_dir)

print(f"FAISS vector store saved to {persist_dir}")

## 2. Semantic Search Pipeline

The following cells demonstrate how to perform semantic search using the prepared vector stores.

**Features**
- Load persisted vector stores (default or FAISS)
- Configure query engines with different parameters
- Run semantic search queries
- Compare results and performance

### Setup and Imports

In [ ]:
import os
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from IPython.display import Markdown, display
import time

### Configure Embedding Model

In [ ]:
# Disable default LLM (we only need embeddings for semantic search)
Settings.llm = None

# Load the same embedding model used during indexing
embedding_model = "all-MiniLM-L6-v2"
Settings.embed_model = HuggingFaceEmbedding(model_name=embedding_model)

print(f"Embedding model loaded: {embedding_model}")

### Load Default Vector Store

In [ ]:
persist_dir_default = os.path.join(res_folder, "sample_data/vector_store/default")

if os.path.exists(persist_dir_default):
    storage_context_default = StorageContext.from_defaults(
        persist_dir=persist_dir_default
    )
    index_default = load_index_from_storage(storage_context_default)
    print(f"✓ Default vector store loaded from {persist_dir_default}")
else:
    print(f"✗ Default vector store not found at {persist_dir_default}")
    index_default = None

### Load FAISS Vector Store

In [ ]:
persist_dir_faiss = os.path.join(res_folder, "sample_data/vector_store/faiss")

if os.path.exists(persist_dir_faiss):
    try:
        from llama_index.vector_stores.faiss import FaissVectorStore
        import faiss

        # FAISS stores the index as binary data in default__vector_store.json
        # We need to load it using faiss.read_index and then reconstruct the vector store
        faiss_index_path = os.path.join(persist_dir_faiss, "default__vector_store.json")

        if os.path.exists(faiss_index_path):
            print(f"Loading FAISS index from: {faiss_index_path}")

            # Load the FAISS index from the binary file
            faiss_index = faiss.read_index(faiss_index_path)
            print(f"✓ FAISS index loaded: {faiss_index.ntotal} vectors")

            # Reconstruct the FaissVectorStore
            vector_store_faiss = FaissVectorStore(faiss_index=faiss_index)

            # Load the storage context with the reconstructed vector store
            storage_context_faiss = StorageContext.from_defaults(
                vector_store=vector_store_faiss, persist_dir=persist_dir_faiss
            )

            # Load the index
            index_faiss = load_index_from_storage(storage_context_faiss)
            print(f"✓ FAISS vector store loaded successfully from {persist_dir_faiss}")
    except ImportError as e:
        print(f"✗ FAISS dependencies not installed: {e}")
        index_faiss = None
    except Exception as e:
        print(f"✗ Error loading FAISS vector store: {e}")
        print(f"  Error type: {type(e).__name__}")
        import traceback

        traceback.print_exc()
        index_faiss = None
else:
    print(f"✗ FAISS vector store not found at {persist_dir_faiss}")
    index_faiss = None

### Create Query Engines

Query engines handle the semantic search process. Key parameters:
- `similarity_top_k`: Number of most similar documents to retrieve
- `response_mode`: How to synthesize the response (we'll use 'no_text' to just get documents)

In [ ]:
# Create query engines with different configurations
query_engines = {}

if index_default:
    query_engines["default"] = index_default.as_query_engine(
        similarity_top_k=5,
        response_mode="no_text",  # Only retrieve documents, no LLM synthesis
    )
    print("✓ Default query engine created (top_k=5)")

if index_faiss:
    query_engines["faiss"] = index_faiss.as_query_engine(
        similarity_top_k=5, response_mode="no_text"
    )
    print("✓ FAISS query engine created (top_k=5)")

print(f"\nAvailable query engines: {list(query_engines.keys())}")

### Helper Functions for Result Display

In [ ]:
def display_search_results(query, response, engine_name=""):
    """
    Display semantic search results in a formatted way.
    """
    title = (
        f"### Search Results: {engine_name}" if engine_name else "### Search Results"
    )
    display(Markdown(title))
    display(Markdown(f"**Query:** {query}"))
    display(Markdown(f"**Results Found:** {len(response.source_nodes)}"))
    display(Markdown("---"))

    for i, node in enumerate(response.source_nodes, start=1):
        # Extract metadata
        metadata = node.node.metadata
        score = node.score if hasattr(node, "score") else "N/A"

        # Display result
        display(
            Markdown(
                f"#### Result #{i} (Score: {score:.4f})"
                if isinstance(score, float)
                else f"#### Result #{i}"
            )
        )
        display(Markdown(f"**URL:** {metadata.get('url', 'N/A')}"))
        display(
            Markdown(
                f"**Snapshot ID:** {metadata.get('snapshot_id', 'N/A')}, **Page Index:** {metadata.get('page_idx', 'N/A')}"
            )
        )

        # Show text preview (first 300 characters)
        text_preview = (
            node.node.text[:300] + "..."
            if len(node.node.text) > 300
            else node.node.text
        )
        display(Markdown(f"**Text Preview:**\n> {text_preview}"))
        display(Markdown("---"))


def compare_search_results(query, engines_dict):
    """
    Run the same query on multiple engines and compare results.
    """
    display(Markdown(f"## Comparing Query: '{query}'"))
    display(Markdown("---"))

    results = {}
    timings = {}

    for engine_name, engine in engines_dict.items():
        start_time = time.time()
        response = engine.query(query)
        elapsed_time = time.time() - start_time

        results[engine_name] = response
        timings[engine_name] = elapsed_time

        display_search_results(query, response, engine_name.upper())

    # Display timing comparison
    display(Markdown("### Performance Comparison"))
    for engine_name, elapsed in timings.items():
        display(Markdown(f"- **{engine_name.upper()}**: {elapsed:.4f} seconds"))

    return results, timings


print("Helper functions loaded successfully!")

### Example Queries

Let's run some semantic search queries to test the pipeline.

#### Query 1: Traffic Light System

In [ ]:
query1 = "What is the traffic light system?"

# Use default engine if available
if "default" in query_engines:
    response1 = query_engines["default"].query(query1)
    display_search_results(query1, response1, "Default Vector Store")
else:
    print("Default vector store not available")

#### Query 2: Vaccination Requirements

In [ ]:
query2 = "vaccination requirements and mandates"

if "default" in query_engines:
    response2 = query_engines["default"].query(query2)
    display_search_results(query2, response2, "Default Vector Store")
else:
    print("Default vector store not available")

#### Query 3: Face Mask Rules

In [ ]:
query3 = "face mask requirements at different alert levels"

if "default" in query_engines:
    response3 = query_engines["default"].query(query3)
    display_search_results(query3, response3, "Default Vector Store")
else:
    print("Default vector store not available")

### Compare Vector Stores

If both vector stores are available, let's compare their performance and results.

In [ ]:
if len(query_engines) > 1:
    comparison_query = "COVID-19 protection framework and alert levels"
    results, timings = compare_search_results(comparison_query, query_engines)
else:
    print(
        f"Only {len(query_engines)} query engine(s) available. Need at least 2 for comparison."
    )
    print(f"Available: {list(query_engines.keys())}")

### Citation

**Title:** Build Vector Store and Semantic Search Pipeline (Jupyter notebook)
**Author:** Yizhe Zhan, Ben O'Brien  
**Affiliation:** National Library of New Zealand, Wellington  
**Last updated:** 2026‑04‑14  

**Contact:** yizhe.git@gmail.com  
**Repository:** https://github.com/NLNZDigitalPreservation/wa-nlnz-toolkit